# 04 - MLflow Tracking：追蹤與 Prompt 管理

本筆記本示範如何整合 MLflow 進行 LLM 呼叫追蹤、指標記錄與 Prompt 模板管理。

## 簡介

MLflow 追蹤整合提供以下功能：

- **追蹤（Tracing）**：自動記錄 LLM 呼叫、工作流節點與工作流的執行 Span，方便除錯與效能分析。
- **日誌記錄（Logging）**：手動記錄 LLM 呼叫詳情、模型參數與效能指標。
- **Prompt 管理（Prompt Management）**：集中管理、載入與渲染 Prompt 模板，支援版本控制與匯出。

框架採用「優雅降級」設計：當 MLflow 服務不可用時，所有追蹤與記錄操作會自動跳過，不影響主要業務邏輯。

In [ ]:
import sys
import os

# 確保 project root 在 Python path 中
sys.path.insert(0, os.path.abspath(".."))

# 載入設定與初始化 MLflow
from app.utils.config import init_config
from app.tracking import init_mlflow, is_mlflow_available, PromptManager
from app.tracking.tracer import trace_llm_call, trace_node, trace_workflow, span
from app.tracking.mlflow_logger import log_llm_call, log_params, log_metrics

# 初始化設定
cfg = init_config()

# 初始化 MLflow
init_mlflow(cfg)

# 確認 MLflow 是否可用
available = is_mlflow_available()
print(f"MLflow 可用狀態: {available}")

## Tracing Decorators：自動追蹤裝飾器

框架提供三種裝飾器，可自動為函式建立 MLflow Span：

| 裝飾器 | 用途 | Span 類型 |
|--------|------|----------|
| `@trace_llm_call` | 標記 LLM 呼叫函式 | `LLM` |
| `@trace_node("name")` | 標記工作流節點函式 | `CHAIN` |
| `@trace_workflow("name")` | 標記整個工作流 | `CHAIN` |

裝飾器會自動記錄函式的輸入參數與回傳值，並在 MLflow UI 中顯示完整的呼叫樹。

In [ ]:
import time

# 使用 @trace_llm_call 裝飾 LLM 呼叫函式
@trace_llm_call
def call_llm(prompt: str, model: str = "gpt-4o") -> str:
    """模擬 LLM 呼叫，實際使用時替換為真實 API 呼叫。"""
    time.sleep(0.1)  # 模擬網路延遲
    return f"[模擬回應] 收到 Prompt：{prompt[:30]}..."

# 使用 @trace_node 裝飾工作流節點
@trace_node("process")
def process_response(raw_response: str) -> dict:
    """處理 LLM 回應並結構化輸出。"""
    return {
        "status": "success",
        "content": raw_response,
        "length": len(raw_response)
    }

# 呼叫已裝飾的函式，MLflow 會自動記錄 Span
response = call_llm("請解釋什麼是大型語言模型？", model="gpt-4o")
print(f"LLM 回應：{response}")

result = process_response(response)
print(f"處理結果：{result}")

## 手動記錄 LLM 呼叫

除了裝飾器之外，也可以使用 `log_llm_call`、`log_params`、`log_metrics` 進行手動記錄：

- **`log_llm_call`**：記錄完整的 LLM 呼叫資訊，包含 Prompt、回應、模型名稱、Token 用量與延遲。
- **`log_params`**：記錄模型參數或任意實驗參數（字典格式）。
- **`log_metrics`**：記錄數值型效能指標（字典格式）。

In [ ]:
# 手動記錄 LLM 呼叫詳情
sample_prompt = "請用繁體中文說明 RAG（檢索增強生成）的工作原理。"
sample_response = "RAG 是一種結合資訊檢索與生成式 AI 的技術..."

log_llm_call(
    prompt=sample_prompt,
    response=sample_response,
    model="gpt-4o",
    token_usage={"prompt_tokens": 42, "completion_tokens": 128, "total_tokens": 170},
    latency_ms=523.7
)
print("LLM 呼叫已記錄")

# 記錄模型參數
log_params({
    "model": "gpt-4o",
    "temperature": 0.7,
    "max_tokens": 512,
    "top_p": 1.0
})
print("參數已記錄")

# 記錄效能指標
log_metrics({
    "latency_ms": 523.7,
    "total_tokens": 170,
    "tokens_per_second": 170 / (523.7 / 1000),
    "cost_usd": 0.00255
})
print("指標已記錄")

## 使用 Span Context Manager

`span("name", span_type="CHAIN")` 是一個 Context Manager，適合用於包裹一段需要追蹤的程式碼區塊，而不需要將其封裝成獨立函式。

常見的 `span_type` 值：
- `"LLM"` — LLM 推論呼叫
- `"CHAIN"` — 處理鏈或工作流步驟
- `"RETRIEVER"` — 資料檢索操作
- `"TOOL"` — 工具呼叫

In [ ]:
import time

# 使用 span context manager 包裹一段處理邏輯
with span("my_op", span_type="CHAIN"):
    print("開始執行自定義操作...")

    # 模擬資料前處理
    with span("preprocess", span_type="CHAIN"):
        raw_data = {"query": "什麼是 LangGraph？", "lang": "zh-TW"}
        processed = {"prompt": f"[語言: {raw_data['lang']}] {raw_data['query']}", "ready": True}
        time.sleep(0.05)
        print(f"前處理完成：{processed}")

    # 模擬 LLM 推論
    with span("inference", span_type="LLM"):
        time.sleep(0.1)  # 模擬 API 延遲
        llm_output = "LangGraph 是 LangChain 提供的狀態圖工作流框架..."
        print(f"推論完成：{llm_output[:40]}...")

    # 模擬後處理
    with span("postprocess", span_type="CHAIN"):
        final_result = {"answer": llm_output, "confidence": 0.95}
        print(f"後處理完成，最終結果已就緒")

print("自定義操作執行完畢，所有 Span 已記錄至 MLflow")

## Prompt 管理

`PromptManager` 提供集中化的 Prompt 模板管理功能：

- **`register(name, template)`**：註冊一個具名 Prompt 模板，模板中使用 `{variable}` 語法定義變數。
- **`load(name)`**：載入已註冊的模板（也可從設定檔或外部儲存載入）。
- **`render(name, **vars)`**：以指定變數渲染 Prompt 模板，回傳完整的 Prompt 字串。
- **`export_all()`**：匯出所有已註冊的 Prompt 模板，方便備份或版本控制。

In [ ]:
# 初始化 PromptManager
pm = PromptManager(cfg)

# 註冊 Prompt 模板
pm.register(
    name="qa_prompt",
    template=(
        "你是一位專業的 {domain} 專家。\n"
        "請用繁體中文回答以下問題：\n\n"
        "問題：{question}\n\n"
        "回答要求：{requirements}"
    )
)

pm.register(
    name="summary_prompt",
    template=(
        "請將以下文字摘要成不超過 {max_words} 字的繁體中文摘要：\n\n"
        "{content}"
    )
)

print("已註冊 Prompt 模板：qa_prompt, summary_prompt")

# 載入模板
template = pm.load("qa_prompt")
print(f"\n載入的模板：\n{template}")

# 渲染 Prompt（帶入變數）
rendered = pm.render(
    "qa_prompt",
    domain="機器學習",
    question="什麼是 Transformer 架構？",
    requirements="簡潔清晰，適合初學者理解，不超過 200 字"
)
print(f"\n渲染後的 Prompt：\n{rendered}")

# 匯出所有模板
all_templates = pm.export_all()
print(f"\n所有已註冊模板：{list(all_templates.keys())}")

## 查看 MLflow UI

執行完追蹤與記錄操作後，可以透過 MLflow UI 視覺化查看所有記錄的實驗、追蹤與指標。

### 啟動方式

在終端機中執行以下指令啟動 MLflow UI 伺服器：

```bash
mlflow ui --port 5000
```

啟動後，開啟瀏覽器並前往：[http://localhost:5000](http://localhost:5000)

### UI 主要功能

| 頁面 | 功能說明 |
|------|----------|
| **Experiments** | 瀏覽所有實驗，比較 Runs 的參數與指標 |
| **Traces** | 查看 LLM 呼叫的完整追蹤樹（Span 層次結構） |
| **Runs** | 查看單一 Run 的詳細參數、指標與 Artifacts |
| **Models** | 管理已註冊的模型版本 |

In [ ]:
# 在 Jupyter 中直接啟動 MLflow UI（背景執行）
# 注意：此指令會在背景啟動伺服器，請在終端機按 Ctrl+C 停止
print("啟動 MLflow UI 伺服器...")
print("指令：mlflow ui --port 5000")
print()
print("UI 位址：http://localhost:5000")
print()
print("提示：")
print("  - 前往 'Traces' 頁籤查看 LLM 呼叫追蹤")
print("  - 前往 'Experiments' 頁籤比較不同 Run 的指標")
print("  - 點擊個別 Run 可查看詳細的參數與 Artifacts")
print()

# 取消下方註解以在背景啟動 MLflow UI
# !mlflow ui --port 5000

## 總結

本筆記本展示了 LLM 框架的 MLflow 追蹤整合能力：

1. **自動追蹤**：使用 `@trace_llm_call`、`@trace_node`、`@trace_workflow` 裝飾器，無需修改業務邏輯即可自動記錄 Span。
2. **手動記錄**：使用 `log_llm_call`、`log_params`、`log_metrics` 精細控制記錄內容。
3. **靈活的 Span 管理**：使用 `span()` Context Manager 包裹任意程式碼區塊。
4. **集中化 Prompt 管理**：使用 `PromptManager` 統一管理、版本化與渲染 Prompt 模板。

### 優雅降級

當 MLflow 服務不可用（例如本地開發環境未啟動 MLflow Server，或設定中 `mlflow.enabled: false`）時，框架會自動跳過所有追蹤與記錄操作，**業務邏輯不受任何影響**。這確保了程式碼在任何環境中都能正常執行，無論 MLflow 是否可用。

```python
# 可隨時使用 is_mlflow_available() 確認狀態
if is_mlflow_available():
    print("MLflow 追蹤已啟用")
else:
    print("MLflow 不可用，以靜默模式執行")
```